In [1]:
import os
os.system("pip install -q lightgbm xgboost scikit-learn joblib")

0

In [2]:
import joblib
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

In [3]:
from google.colab import files

print("📂 Upload stack_model.pkl and scaler.pkl")
uploaded = files.upload()   # select BOTH files at once

📂 Upload stack_model.pkl and scaler.pkl


Saving stack_model.pkl to stack_model.pkl
Saving scaler.pkl to scaler.pkl


In [4]:
# ── 4. LOAD MODEL & SCALER ─────────────────────────────────────
stack_model = joblib.load("stack_model.pkl")
scaler      = joblib.load("scaler.pkl")
print("✅ Model and scaler loaded successfully!")

✅ Model and scaler loaded successfully!


In [5]:
# ── 5. FEATURE COLUMNS (must match training exactly) ───────────
FEATURE_COLS = [
    "age", "gender", "height", "weight", "ap_hi", "ap_lo",
    "smoke", "alco", "active", "bmi", "hypertension",
    "pulse_pressure", "age_bmi_interaction", "bp_ratio",
    "bmi_age_ratio", "chol_gluc_product", "lifestyle_score",
    "cholesterol_2", "cholesterol_3", "gluc_2", "gluc_3"
]

LABEL_MAP = {0: "Low Risk", 1: "Medium Risk", 2: "High Risk"}


In [6]:
# ── 6. INPUT BUILDER ───────────────────────────────────────────
def build_input(age, gender, height, weight,
                ap_hi, ap_lo, cholesterol, gluc,
                smoke, alco, active):
    """
    Accepts raw clinical values and computes all engineered features.

    Parameters
    ----------
    age         : int   — age in YEARS
    gender      : int   — 1=female, 2=male
    height      : int   — cm
    weight      : float — kg
    ap_hi       : int   — systolic blood pressure (mmHg)
    ap_lo       : int   — diastolic blood pressure (mmHg)
    cholesterol : int   — 1=normal, 2=above normal, 3=well above normal
    gluc        : int   — 1=normal, 2=above normal, 3=well above normal
    smoke       : int   — 0 or 1
    alco        : int   — 0 or 1
    active      : int   — 0 or 1 (physical activity)
    """
    bmi = weight / ((height / 100) ** 2)

    row = {
        # raw features
        "age"                : age,
        "gender"             : gender,
        "height"             : height,
        "weight"             : weight,
        "ap_hi"              : ap_hi,
        "ap_lo"              : ap_lo,
        "smoke"              : smoke,
        "alco"               : alco,
        "active"             : active,

        # engineered features (must mirror training pipeline)
        "bmi"                : bmi,
        "hypertension"       : int(ap_hi >= 140 or ap_lo >= 90),
        "pulse_pressure"     : ap_hi - ap_lo,
        "age_bmi_interaction": age * bmi,
        "bp_ratio"           : ap_hi / (ap_lo + 1),
        "bmi_age_ratio"      : bmi / (age + 1),
        "chol_gluc_product"  : cholesterol * gluc,
        "lifestyle_score"    : smoke + alco - active,

        # one-hot encoded (drop_first=True was used in training)
        "cholesterol_2"      : int(cholesterol == 2),
        "cholesterol_3"      : int(cholesterol == 3),
        "gluc_2"             : int(gluc == 2),
        "gluc_3"             : int(gluc == 3),
    }

    df = pd.DataFrame([row]).reindex(columns=FEATURE_COLS, fill_value=0)
    return df


In [7]:
# ── 7. PREDICTION FUNCTION ─────────────────────────────────────
def predict_risk(age, gender, height, weight,
                 ap_hi, ap_lo, cholesterol, gluc,
                 smoke, alco, active,
                 verbose=True):
    """
    Returns
    -------
    dict with keys:
        predicted_class : str   — "Low Risk" / "Medium Risk" / "High Risk"
        predicted_label : int   — 0 / 1 / 2
        confidence      : float — probability of predicted class (0–1)
        probabilities   : dict  — {class_name: probability} for all 3 classes
        fusion_ready    : dict  — clean payload for downstream fusion/API use
    """
    # Build & scale input
    df     = build_input(age, gender, height, weight,
                         ap_hi, ap_lo, cholesterol, gluc,
                         smoke, alco, active)
    scaled = scaler.transform(df)

    # ── Probability output ─────────────────────────────────────
    proba      = stack_model.predict_proba(scaled)[0]   # shape: (3,)
    pred_label = int(np.argmax(proba))
    pred_class = LABEL_MAP[pred_label]
    confidence = float(proba[pred_label])

    # Verify probabilities sum to 1 (sanity check)
    prob_sum = proba.sum()
    assert abs(prob_sum - 1.0) < 1e-4, f"Probabilities don't sum to 1: {prob_sum}"

    # ── Structured probability output ─────────────────────────
    probabilities = {
        "Low Risk"    : round(float(proba[0]), 6),
        "Medium Risk" : round(float(proba[1]), 6),
        "High Risk"   : round(float(proba[2]), 6),
    }

    # ── Fusion-ready payload ───────────────────────────────────
    fusion_ready = {
        "predicted_class" : pred_class,
        "predicted_label" : pred_label,
        "confidence"      : round(confidence, 6),
        "prob_low"        : probabilities["Low Risk"],
        "prob_medium"     : probabilities["Medium Risk"],
        "prob_high"       : probabilities["High Risk"],
        "calibrated"      : True,   # SVC was CalibratedClassifierCV
    }

    # ── Verbose display ────────────────────────────────────────
    if verbose:
        print("\n" + "="*50)
        print("       CARDIOVASCULAR RISK PREDICTION")
        print("="*50)
        print(f"  Predicted Risk Class : {pred_class}")
        print(f"  Confidence           : {confidence:.2%}")
        print("-"*50)
        print("  Class Probabilities:")
        for cls, p in probabilities.items():
            bar = "█" * int(p * 30)
            print(f"    {cls:<14} : {p:.4f}  {bar}")
        print("-"*50)
        print(f"  ✅ Probabilities sum : {prob_sum:.6f}")

        print("="*50)

    return {
        "predicted_class" : pred_class,
        "predicted_label" : pred_label,
        "confidence"      : confidence,
        "probabilities"   : probabilities,
        "fusion_ready"    : fusion_ready,
    }



In [8]:
# ── 8. BATCH PREDICTION (for multiple patients) ────────────────
def predict_batch(patient_list, verbose=False):
    """
    patient_list : list of dicts, each with keys matching predict_risk() args
    Returns      : DataFrame with one row per patient
    """
    results = []
    for i, p in enumerate(patient_list):
        result = predict_risk(**p, verbose=verbose)
        row = {"patient_id": i, **result["fusion_ready"]}
        results.append(row)

    df_results = pd.DataFrame(results)
    return df_results


In [9]:
# ── 9. TEST — SINGLE PATIENT ───────────────────────────────────
print("\n🔬 TEST 1: High-risk patient")
result = predict_risk(
    age=60, gender=2, height=170, weight=95,
    ap_hi=180, ap_lo=110,
    cholesterol=3, gluc=2,
    smoke=1, alco=1, active=0
)

print("\n🔬 TEST 2: Low-risk patient")
result2 = predict_risk(
    age=30, gender=1, height=165, weight=60,
    ap_hi=115, ap_lo=75,
    cholesterol=1, gluc=1,
    smoke=0, alco=0, active=1
)

print("\n🔬 TEST 3: Borderline patient")
result3 = predict_risk(
    age=45, gender=2, height=175, weight=82,
    ap_hi=135, ap_lo=88,
    cholesterol=2, gluc=1,
    smoke=0, alco=1, active=1
)




🔬 TEST 1: High-risk patient

       CARDIOVASCULAR RISK PREDICTION
  Predicted Risk Class : High Risk
  Confidence           : 83.41%
--------------------------------------------------
  Class Probabilities:
    Low Risk       : 0.0077  
    Medium Risk    : 0.1582  ████
    High Risk      : 0.8341  █████████████████████████
--------------------------------------------------
  ✅ Probabilities sum : 1.000000
  ✅ Ready for fusion  : True

🔬 TEST 2: Low-risk patient

       CARDIOVASCULAR RISK PREDICTION
  Predicted Risk Class : Low Risk
  Confidence           : 95.24%
--------------------------------------------------
  Class Probabilities:
    Low Risk       : 0.9524  ████████████████████████████
    Medium Risk    : 0.0465  █
    High Risk      : 0.0010  
--------------------------------------------------
  ✅ Probabilities sum : 1.000000
  ✅ Ready for fusion  : True

🔬 TEST 3: Borderline patient

       CARDIOVASCULAR RISK PREDICTION
  Predicted Risk Class : Medium Risk
  Confidence  

In [10]:
# ── 10. BATCH TEST ─────────────────────────────────────────────
print("\n📋 BATCH PREDICTION TEST")
patients = [
    dict(age=60, gender=2, height=170, weight=95, ap_hi=180, ap_lo=110,
         cholesterol=3, gluc=2, smoke=1, alco=1, active=0),
    dict(age=30, gender=1, height=165, weight=60, ap_hi=115, ap_lo=75,
         cholesterol=1, gluc=1, smoke=0, alco=0, active=1),
    dict(age=45, gender=2, height=175, weight=82, ap_hi=135, ap_lo=88,
         cholesterol=2, gluc=1, smoke=0, alco=1, active=1),
    dict(age=70, gender=1, height=158, weight=78, ap_hi=160, ap_lo=95,
         cholesterol=3, gluc=3, smoke=0, alco=0, active=0),
]

batch_df = predict_batch(patients)
print(batch_df.to_string(index=False))




📋 BATCH PREDICTION TEST
 patient_id predicted_class  predicted_label  confidence  prob_low  prob_medium  prob_high  calibrated
          0       High Risk                2    0.834062  0.007747     0.158191   0.834062        True
          1        Low Risk                0    0.952445  0.952445     0.046541   0.001013        True
          2     Medium Risk                1    0.594253  0.403369     0.594253   0.002378        True
          3       High Risk                2    0.824335  0.007410     0.168255   0.824335        True


In [11]:
# ── 11. EXPORT BATCH RESULTS ───────────────────────────────────
batch_df.to_csv("predictions.csv", index=False)
files.download("predictions.csv")
print("\n✅ predictions.csv downloaded!")

# ── 12. FUSION-READY OUTPUT (single patient) ───────────────────
print("\n🔗 FUSION-READY PAYLOAD (use this to connect to other models):")
import json
print(json.dumps(result["fusion_ready"], indent=2))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ predictions.csv downloaded!

🔗 FUSION-READY PAYLOAD (use this to connect to other models):
{
  "predicted_class": "High Risk",
  "predicted_label": 2,
  "confidence": 0.834062,
  "prob_low": 0.007747,
  "prob_medium": 0.158191,
  "prob_high": 0.834062,
  "calibrated": true
}
